# Session 4 — Building a Gallery

### AI Image Studio · Session 4 of 8

Every image you have made so far has been temporary. Restart the kernel and they're gone.
That's fine for experimenting, and completely useless for making something.

Today we make them permanent, and we do it properly — not `image1.png`, `image2.png`,
`image_final.png`, `image_final_REAL.png`, but filenames that tell you what's inside them
before you open them.

## TODAY:

- **Part A:** bytes, files, and what "binary" really means
- **Part B:** turning a prompt into a good filename
- **Part C:** a loop that fills the gallery

In [ ]:
# imports

import os
import re
import base64
from pathlib import Path
from datetime import date
from dotenv import load_dotenv
from IPython.display import Image, display
from langchain.chat_models import init_chat_model

In [ ]:
# Setup

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
print(f"✅ Key loaded, begins {api_key[:8]}" if api_key else "❌ No key found - check your .env")

CHAT_MODEL = "openai:gpt-4.1-mini"
IMAGE_TOOL = {"type": "image_generation", "quality": "low"}

In [ ]:
# Everything we built in sessions 2 and 3, brought forward

image_model = init_chat_model(CHAT_MODEL).bind_tools([IMAGE_TOOL])

SIZES = {"square": "1024x1024", "wide": "1536x1024", "tall": "1024x1536"}

STYLES = {
    "photo": "photorealistic, natural lighting, shot on 35mm film, shallow depth of field",
    "watercolour": "watercolour painting, soft edges, muted colours, visible paper texture",
    "pixel": "16-bit pixel art, limited colour palette, crisp pixels, retro game sprite",
    "cinematic": "cinematic film still, dramatic lighting, wide aspect, moody atmosphere",
    "sketch": "rough pencil sketch, black and white, visible construction lines",
    "cute": "cute flat vector illustration, bold outlines, bright friendly colours",
}


def extract_image_bytes(response):
    """Find the image inside a model's response and return it as raw bytes."""
    for block in response.content_blocks:
        if block["type"] == "image":
            return base64.b64decode(block["base64"])
    raise ValueError("The model didn't return an image. What did it say instead?")


def make_image(prompt, style_name="photo", size="square"):
    """Generate an image using a named style, and return it as bytes."""
    response = image_model.invoke(
        f"{prompt}. {STYLES[style_name]}. Image size: {SIZES[size]}."
    )
    return extract_image_bytes(response)

## PART A: Bytes and files

In [ ]:
# Make one image to work with today

image_bytes = make_image("a paper boat floating down a rainy street", "cinematic")
display(Image(image_bytes))

### What is `image_bytes`, really?

Let's look at it instead of assuming.

In [ ]:
# What kind of thing is it?

type(image_bytes)

In [ ]:
# How big is it?

f"{len(image_bytes):,} bytes  ({len(image_bytes) / 1024:.0f} KB)"

In [ ]:
# And what's actually in there? Look at the first 20 bytes.

image_bytes[:20]

### Reading that output

You'll see something like `b'\x89PNG\r\n\x1a\n\x00...'`.

That `b` at the front means **bytes**, not text. And look at what's right at the
beginning: the letters `PNG`.

Those first few bytes are called a **magic number** — a fixed marker at the start of a file
that says what kind of file it is. It's how your operating system knows a file is a PNG even
if you rename it to `homework.txt`. The file extension is a hint for humans; the magic
number is the truth.

In [ ]:
# Proof. Every PNG file in the world starts with these exact 8 bytes.

image_bytes[:8] == b"\x89PNG\r\n\x1a\n"

### Writing it to disk

In [ ]:
# "wb" = write, binary. The b is essential - without it Python would try to
# treat this as text, mangle it, and produce a corrupt file.

Path("test_image.png").write_bytes(image_bytes)

print("Saved. Look in the file list on the left - you can click it to open it.")

<div style="border-left: 6px solid #06c; background: #f2f7ff; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#06c; margin-top:0;">🔄 Meet <code>Path</code></h3>
<p style="color:#06c; margin-bottom:0;">
<code>Path("test_image.png")</code> makes a <b>path object</b> — Python's way of talking
about a location on disk.<br/><br/>
You could join folder names with <code>"/"</code> yourself, but that breaks on Windows,
which uses <code>\</code>. <code>Path</code> handles that for you, so the same code works on
every student's laptop regardless of what they're running.<br/><br/>
It has some nice methods: <code>.exists()</code>, <code>.write_bytes()</code>,
<code>.read_bytes()</code>, <code>.mkdir()</code>, and a <code>/</code> operator for joining:
<code>Path("gallery") / "cat.png"</code>.
</p>
</div>

In [ ]:
# Joining paths with / . This reads beautifully and works on every operating system.

Path("gallery") / "my_picture.png"

## PART B: Filenames that tell you something

Here's the problem. This is a real prompt:

> `"a ginger cat asleep on a radiator, soft evening light!!"`

And here's what we can't use it as: a filename. It has spaces, commas, and exclamation
marks, and on some systems that's a broken file or a security hole.

So we need to turn a prompt into a **slug** — a short, safe, lowercase version.

### Building the slug step by step

We'll do it in a scratch cell first and watch each step, then wrap it up.

In [ ]:
prompt = "A ginger cat asleep on a radiator, soft evening light!!"

# Step 1: lowercase
step1 = prompt.lower()
step1

In [ ]:
# Step 2: throw away anything that isn't a letter, a number, or a space.
# re.sub means "substitute": find everything matching this pattern, replace it with "".

step2 = re.sub(r"[^a-z0-9 ]", "", step1)
step2

In [ ]:
# Step 3: cut it down to the first five words, so filenames stay short

step3 = " ".join(step2.split()[:5])
step3

In [ ]:
# Step 4: spaces become hyphens

step4 = step3.replace(" ", "-")
step4

<div style="border-left: 6px solid #06c; background: #f2f7ff; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#06c; margin-top:0;">🔄 What <code>" ".join(text.split()[:5])</code> is doing</h3>
<p style="color:#06c; margin-bottom:0;">
Read it from the inside out, which is how you read any nested expression:<br/><br/>
1. <code>step2.split()</code> — break the string into a list of words (JavaScript:
<code>.split(" ")</code>)<br/>
2. <code>[:5]</code> — take the first five. This is <b>slicing</b>, and it's one of the
nicest things in Python<br/>
3. <code>" ".join(...)</code> — glue that list back into one string with spaces between.
JavaScript writes this as <code>array.join(" ")</code>; Python puts the glue first, which
looks backwards until it doesn't
</p>
</div>

### Now wrap it up

In [ ]:
def slugify(text, max_words=5):
    """
    Turn a prompt into something safe to use as a filename.

    "A ginger cat asleep on a radiator!!" -> "a-ginger-cat-asleep-on"

    We strip out punctuation because some characters are illegal in filenames on some
    operating systems, and we cut the length because nobody wants a 200-character filename.
    """
    text = text.lower()
    text = re.sub(r"[^a-z0-9 ]", "", text)      # keep only letters, digits and spaces
    words = text.split()[:max_words]            # first few words only
    return "-".join(words)

In [ ]:
# Try it on a few things, including nasty ones

for test in [
    "A ginger cat asleep on a radiator, soft evening light!!",
    "лодка",
    "!!!???",
    "a lighthouse",
]:
    print(f"{test!r:60} -> {slugify(test)!r}")

Notice the last two. `"!!!???"` gives an **empty** slug, and a non-English prompt loses
everything, because our pattern only keeps `a-z`. That's a real bug, and we'll fix it in a
moment — but first notice how we found it: by deliberately testing the awkward cases rather
than only the one we had in mind.

In [ ]:
def slugify(text, max_words=5):
    """
    Turn a prompt into something safe to use as a filename.

    Falls back to "image" when there's nothing usable left - an empty filename would
    produce a file called just ".png", which is invisible on Mac and Linux.
    """
    text = text.lower()
    text = re.sub(r"[^a-z0-9 ]", "", text)
    words = text.split()[:max_words]
    return "-".join(words) or "image"    # `or` supplies the fallback when the slug is empty

In [ ]:
slugify("!!!???")

<div style="border-left: 6px solid #06c; background: #f2f7ff; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#06c; margin-top:0;">🔄 That <code>or</code> at the end</h3>
<p style="color:#06c; margin-bottom:0;">
<code>"-".join(words) or "image"</code> means "use the join, unless it's empty, in which
case use <code>"image"</code>".<br/><br/>
This is exactly JavaScript's <code>||</code>. Python spells it <code>or</code>, and an
empty string counts as false in both languages.
</p>
</div>

### The saving function

In [ ]:
GALLERY = Path("gallery")


def save_image(image_bytes, prompt, style_name):
    """
    Save an image into the gallery folder with a filename that describes it.

    Produces something like:  gallery/2026-03-14_a-paper-boat-floating-down_cinematic.png

    The date goes first so that sorting the folder by name also sorts it by time,
    and the prompt and style are in there so you can find an image again without
    opening thirty files.

    Returns the Path it saved to, so the caller can tell the user where it went.
    """
    # exist_ok=True means "don't complain if the folder is already there"
    GALLERY.mkdir(exist_ok=True)

    filename = f"{date.today()}_{slugify(prompt)}_{style_name}.png"
    path = GALLERY / filename
    path.write_bytes(image_bytes)
    return path

In [ ]:
# Try it end to end

prompt = "a paper boat floating down a rainy street"
image_bytes = make_image(prompt, "cinematic")
path = save_image(image_bytes, prompt, "cinematic")

print(f"Saved to {path}")
display(Image(image_bytes))

In [ ]:
# What's in the gallery so far? sorted() puts them in order.

for f in sorted(GALLERY.iterdir()):
    print(f.name)

<div style="border-left: 6px solid #181; background: #f3fbf3; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#181; margin-top:0;">🌍 Where you'll see this in the real world</h3>
<p style="color:#181; margin-bottom:0;">
Putting information <i>into the filename</i> — a date that sorts correctly, a description,
a variant name — is how photographers, video editors, scientists and engineers all survive
folders with ten thousand files in them.<br/><br/>
The date format matters too. <code>2026-03-14</code> sorts correctly as plain text;
<code>14-03-2026</code> does not. That's why this format is an international standard
(ISO 8601), and why you should use it for the rest of your life.
</p>
</div>

## PART C: Filling the gallery with a loop

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🛑 Stop and count before you run</h3>
<p style="color:#900; margin-bottom:0;">
The next cell makes <b>one image per prompt in the list</b>. Four prompts, four images,
about a minute, a few cents.<br/><br/>
Before you run <i>any</i> loop that calls an API, ask yourself: <b>how many calls is this?</b>
If you put a list of 4 prompts inside a loop over 6 styles, that's not 10 images — it's 24.
<br/><br/>
This is the single most common way people get a surprise bill, at school and at work alike.
Count first. Every time.
</p>
</div>

In [ ]:
# Four prompts. Count them: four images.

prompts = [
    "a paper boat floating down a rainy street",
    "an old bookshop with a cat in the window",
    "a bicycle abandoned in a field of tall grass",
    "a lighthouse keeper drinking tea at dawn",
]

len(prompts)

In [ ]:
# Now generate and save each one.
# `enumerate` gives us both the position and the item, so we can print progress -
# without it you stare at a blank cell for a minute wondering if it's broken.

for i, p in enumerate(prompts, start=1):
    print(f"[{i}/{len(prompts)}] {p}")
    image_bytes = make_image(p, "watercolour")
    path = save_image(image_bytes, p, "watercolour")
    print(f"          saved to {path}")

print("\nAll done!")

In [ ]:
# Look at the whole gallery

for f in sorted(GALLERY.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"{f.name:65} {size_kb:6.0f} KB")

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🎯 Now try it yourself</h3>
<p style="color:#900; margin-bottom:0;">
Make a <b>set</b> of images that belong together — four prompts that share a theme, all in
the same style. A tiny story, four seasons of one place, four characters from something
you like.<br/><br/>
Then open the gallery folder in your file browser and look at them as a group. Do they feel
like a set? If not, which prompt is the odd one out, and is it the prompt's fault or the
style's?
</p>
</div>

In [ ]:
# Your set. Keep it to four - count before you run.

my_prompts = [
    "",
    "",
    "",
    "",
]

my_style = "watercolour"   # change to whichever you like, including the one you invented

for i, p in enumerate(my_prompts, start=1):
    print(f"[{i}/{len(my_prompts)}] {p}")
    image_bytes = make_image(p, my_style)
    print(f"          saved to {save_image(image_bytes, p, my_style)}")

<div style="border-left: 6px solid #f71; background: #fff7f0; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#f71; margin-top:0;">🚀 Extra challenge</h3>
<p style="color:#f71; margin-bottom:0;">
Two of them, pick either:<br/><br/>
<b>1.</b> Right now, generating the same prompt in the same style twice <b>overwrites</b>
the first file, because the filename is identical. Fix it — if the file already exists, add
<code>-2</code>, then <code>-3</code>. <i>Hint:</i> <code>path.exists()</code> and a
<code>while</code> loop.<br/><br/>
<b>2.</b> Write <code>save_image</code>'s companion: a function that writes a
<code>gallery/index.md</code> listing every image with its prompt, so you have a
contact sheet you can read.
</p>
</div>

## Where we got to

- Bytes are not text, `"wb"` is not optional, and a PNG announces itself in its first 8 bytes
- `Path` for locations, because string-joining paths breaks on Windows
- `slugify` — built one step at a time, then tested on inputs designed to break it
- Filenames carrying date + prompt + style, with the date first so sorting works
- A counted loop, and the habit of counting *before* running

## Next session

Our prompts are only as good as we are at writing them. Next session we add a **second
model** whose entire job is to take a lazy prompt and make it vivid — and that's also where
we discover how to swap the whole thing to a different AI company, or to a model running on
this laptop with the wifi switched off.